Module 4 — Stage 1: domain-adaptation fine-tune (BART-base on QMSum)

Run this in Google Colab (Runtime > Change runtime type > T4 GPU).
Paste each "# %% CELL" block into its own Colab cell, in order.

What this teaches the model: compress a chunk of spoken transcript into a
short factual summary (QMSum's specific_query_list = local segment -> local
summary, structurally the closest public analogue to "slide's audio window
-> slide summary"). It does NOT yet teach the fused_slides JSON schema
(title/summary/key_concepts/code_example/voiceover_script) or use any real
lecture_021 data — that's Stage 2, once a small bootstrapped example set
exists from the real lecture files.

In [ ]:
# %% CELL 1 — install packages
!pip install -q "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.30.0" "evaluate>=0.4.0" "rouge_score" "sentencepiece"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
# %% CELL 2 — GPU check
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "CPU only — go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
Device: Tesla T4


In [ ]:
# %% CELL 3 — get QMSum (public repo, no auth needed)
!git clone https://github.com/Yale-LILY/QMSum.git

Cloning into 'QMSum'...
remote: Enumerating objects: 809, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 809 (delta 1), reused 0 (delta 0), pack-reused 804 (from 1)
Receiving objects: 100% (809/809), 13.76 MiB | 14.90 MiB/s, done.
Resolving deltas: 100% (446/446), done.
Updating files: 100% (718/718), done.


In [ ]:
# %% CELL 3.5 — install gdown for data download
!pip install gdown -q

In [ ]:
# %% CELL 3.6 — download QMSum data
# The QMSum dataset's `train`, `val`, and `test` folders are located in this shared Google Drive folder.
# We will download them directly into the QMSum/data/ALL directory.

# First, navigate to the target directory where the data should reside
%cd QMSum/data/ALL

# Google Drive folder ID for the QMSum dataset's train/val/test subfolders
QMSUM_DATA_FOLDER_ID = '1G8VzXm9Zz3E2v3F1K4y-h0C8o3O1uQ0o'

# Download the content of the folder. `gdown` will create 'train', 'val', 'test' subdirectories
# and place their respective .jsonl files inside them.
!gdown --folder $QMSUM_DATA_FOLDER_ID

# Navigate back to the original content directory
%cd /content/

# Verify that the files are now present in the correct structure
!ls -R QMSum/data/ALL/

/content/QMSum/data/ALL
Retrieving folder contents
Failed to retrieve folder contents
/content
QMSum/data/ALL/:
jsonl  test  train  val

QMSum/data/ALL/jsonl:
test.jsonl  train.jsonl  val.jsonl

QMSum/data/ALL/test:
Bed003.json  Bro019.json	education_9.json  ES2011c.json	TS3004b.json
Bed008.json  Bro027.json	ES2004a.json	  ES2011d.json	TS3004c.json
Bed016.json  covid_4.json	ES2004b.json	  IS1003a.json	TS3004d.json
Bmr006.json  covid_9.json	ES2004c.json	  IS1003b.json	TS3011a.json
Bmr014.json  education_13.json	ES2004d.json	  IS1003c.json	TS3011b.json
Bmr023.json  education_17.json	ES2011a.json	  IS1003d.json	TS3011c.json
Bro004.json  education_4.json	ES2011b.json	  TS3004a.json	TS3011d.json

QMSum/data/ALL/train:
Bdb001.json  Bro018.json	ES2002b.json  ES2014c.json  IS1008b.json
Bed004.json  Bro021.json	ES2002c.json  ES2014d.json  IS1008c.json
Bed005.json  Bro023.json	ES2002d.json  ES2015a.json  IS1008d.json
Bed006.json  Bro024.json	ES2003a.json  ES2015b.json  IS1009a.json
Bed009.json  

In [ ]:
import json, os

DATA_DIR = "QMSum/data/ALL"

def find_file(keyword):
    # Based on the `ls -R` output, the .jsonl files are in a 'jsonl' subdirectory
    # within DATA_DIR.
    expected_path = os.path.join(DATA_DIR, "jsonl", f"{keyword}.jsonl")

    if os.path.isfile(expected_path):
        return expected_path
    else:
        raise FileNotFoundError(
            f"Expected data file not found: '{expected_path}'. "
            f"Please ensure the QMSum dataset is correctly cloned "
            f"and has files like 'QMSum/data/ALL/jsonl/{keyword}.jsonl'."
        )

TRAIN_FILE = find_file("train")
VAL_FILE = find_file("val")
print("train file:", TRAIN_FILE)
print("val file:", VAL_FILE)

def load_meetings(path):
    meetings = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                meetings.append(json.loads(line))
    return meetings

def spans_to_text(transcript, spans):
    parts = []
    for span in spans:
        start, end = int(span[0]), int(span[1])
        for turn in transcript[start:end + 1]:
            speaker = (turn.get("speaker") or "").strip()
            content = (turn.get("content") or "").strip()
            if content:
                parts.append(f"{speaker}: {content}" if speaker else content)
    return "\n".join(parts)

def build_examples(meetings):
    examples = []
    for meeting in meetings:
        transcript = meeting.get("meeting_transcripts", [])
        for q in meeting.get("specific_query_list", []):
            answer = (q.get("answer") or "").strip()
            spans = q.get("relevant_text_span", [])
            if not answer or not spans:
                continue
            segment_text = spans_to_text(transcript, spans)
            if not segment_text:
                continue
            input_text = (
                "Summarize this lecture/meeting segment in 2-4 sentences, "
                "capturing only what was actually discussed.\n\n"
                f"Transcript segment:\n{segment_text}"
            )
            examples.append({"input": input_text, "target": answer})
    return examples

train_meetings = load_meetings(TRAIN_FILE)
val_meetings = load_meetings(VAL_FILE)

train_examples = build_examples(train_meetings)
val_examples = build_examples(val_meetings)

print(f"train examples: {len(train_examples)}")
print(f"val examples:   {len(val_examples)}")
print("\n--- sample ---")
print(train_examples[0]["input"][:500])
print("---")
print(train_examples[0]["target"])

train file: QMSum/data/ALL/jsonl/train.jsonl
val file: QMSum/data/ALL/jsonl/val.jsonl
train examples: 1095
val examples:   237

--- sample ---
Summarize this lecture/meeting segment in 2-4 sentences, capturing only what was actually discussed.

Transcript segment:
Project Manager: Yep . Soon as I get this . Okay . This is our last meeting . Um I'll go ahead and go through the minutes from the previous meeting . Uh and then we'll have a , the prototype presentation . {vocalsound} Um then we will um do an evaluation . Uh or we'll see what , what we need to have under the criteria for the evaluation . Then we'll go through the finance and
---
Project Manager introduced that the prototype incorporated fashion trends that people prefer fancy looking products like fruit and vegetable. After That, User Interface presented the product which looked like a banana and was bright yellow except for the blue button. The style was as simple as possible in order to fit the customers' need for simplici

In [ ]:
# %% CELL 5 — build HF datasets
from datasets import Dataset, DatasetDict

raw_datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "validation": Dataset.from_list(val_examples),
})
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 1095
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 237
    })
})

In [ ]:
# %% CELL 6 — tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-base"
MAX_INPUT_LEN = 1024
MAX_TARGET_LEN = 160

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    model_inputs = tokenizer(batch["input"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw_datasets.map(preprocess, batched=True, remove_columns=["input", "target"])

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/1095 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

In [ ]:
# %% CELL 7 — ROUGE metric for eval
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

In [ ]:
# %% CELL 8 — train
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/module4_stage1_checkpoints",
    eval_strategy="epoch",          # older transformers: rename to evaluation_strategy
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    # tokenizer=tokenizer, # Remove this line as tokenizer is handled by data_collator
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,12.645226,2.811053,19.740000,6.710000,15.430000,15.420000
2,11.110029,2.742570,19.610000,6.800000,15.490000,15.510000
3,10.473628,2.728703,19.720000,6.850000,15.740000,15.710000


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=207, training_loss=11.163356246579673, metrics={'train_runtime': 359.1078, 'train_samples_per_second': 9.148, 'train_steps_per_second': 0.576, 'total_flos': 1943134821089280.0, 'train_loss': 11.163356246579673, 'epoch': 3.0})

In [ ]:
# %% CELL 9 — save, zip, download
SAVE_DIR = "/content/module4_stage1_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.make_archive("/content/module4_stage1_model", "zip", SAVE_DIR)

from google.colab import files
files.download("/content/module4_stage1_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Check the disk size of the saved model
!du -sh {SAVE_DIR}
!du -sh {SAVE_DIR}.zip

536M	/content/module4_stage1_model
494M	/content/module4_stage1_model.zip
